# Métricas de evaluación

In [1]:
import numpy as np


### Precisión 

In [48]:
def precision(l: list) -> float:
    """
    Calcula la precisión de los documentos recuperados:

        P = |A ∩ Rel(q)| / |A|

    Por lo tanto, la función calcula la proporción de documentos relevantes
    dentro del conjunto de documentos recuperados.

    Parameters
    ----------
    l : list
        Vector binario.

    Returns
    -------
    float
        Precisión calculada como la proporción de documentos relevantes
        recuperados.

        Retorna 0.0 para el siguiente edge case:
        - el vector está vacío.

    """
    l = np.asarray(l, dtype=float)

    if l.size == 0:
        return 0.0

    return float(np.mean(l))


relevance_query = [0, 1, 0, 0, 1]
print("Ejemplo:", precision(relevance_query))

test_1 = [1, 1, 0, 1, 0]
test_2 = [1, 1, 1, 1]

print("Prueba 1:", precision(test_1))
print("Prueba 2:", precision(test_2))
print("Edge case - vector vacío:", precision([]))


Ejemplo: 0.4
Prueba 1: 0.6
Prueba 2: 1.0
Edge case - vector vacío: 0.0


### Precision at K


In [47]:
def precision_at_k(l: list, k: int) -> float:
    """
    Sean A_k los primeros k documentos recuperados y Rel(q) el conjunto de
    documentos relevantes para la consulta. Precision@K se define como:

        P@k = |A_k ∩ Rel(q)| / k

    Por lo tanto, para un k válido, la función cuenta cuántos documentos
    relevantes aparecen entre las primeras k posiciones y divide este
    número entre k.

    Parameters
    ----------
    l : list
        Vector binario

    k : int
        Número de primeras posiciones del ranking que se evaluarán.

    Returns
    -------
    float
        P@k calculada como el número de documentos relevantes
        encontrados entre las primeras k posiciones dividido entre k.

        Retorna 0.0 para los siguientes edge cases:
        - el vector está vacío;
        - k es menor o igual que cero;
        - k es mayor que la longitud del vector;
        
    """
    l = np.asarray(l, dtype=float)

    if l.size == 0 or k <= 0 or k > l.size:
        return 0.0

    return float(np.sum(l[:k]) / k)


relevance_query = [0, 1, 0, 0, 1]
k = 3
print("Ejemplo:", precision_at_k(relevance_query, k))

test_1 = [1, 1, 0, 1, 0]
test_2 = [0, 0, 0, 0, 0]

print("Prueba propia 1:", precision_at_k(test_1, 3))
print("Prueba propia 2:", precision_at_k(test_2, 3))
print("Edge case - vector vacío:", precision_at_k([], 3))
print("Edge case - k mayor que la longitud:", precision_at_k([1, 0, 1], 5))



Ejemplo: 0.3333333333333333
Prueba propia 1: 0.6666666666666666
Prueba propia 2: 0.0
Edge case - vector vacío: 0.0
Edge case - k mayor que la longitud: 0.0


### Recall at K 

In [29]:
def recall_at_k(l: list,number_relevant_docs: int,k: int) -> float:
    """
    A_k los primeros k documentos recuperados y Rel(q) el conjunto de
    documentos relevantes para la consulta. Recall@K se define como:

        R@k = |A_k ∩ Rel(q)| / |Rel(q)|

    Por lo tanto, para un k válido, la función cuenta cuántos documentos
    relevantes aparecen entre las primeras k posiciones y divide este
    número entre el total de documentos relevantes para la consulta.

    Parameters
    ----------
    l : list
        Vector binario.

    number_relevant_docs : int
        Número total de documentos relevantes para la consulta.

    k : int
        Número de primeras posiciones del ranking que se evaluarán.

    Returns
    -------
    float
        R@k calculado como el número de documentos relevantes encontrados
        entre las primeras k posiciones dividido entre el total de documentos
        relevantes para la consulta.

        Retorna 0.0 para los siguientes edge cases:
        - el vector está vacío;
        - k es menor o igual que cero;
        - k es mayor que la longitud del vector;
        - la consulta no tiene documentos relevantes.

    """
    l = np.asarray(l, dtype=float)

    if (l.size == 0 or k <= 0 or k > l.size or number_relevant_docs <= 0):
        return 0.0

    return float(np.sum(l[:k]) / number_relevant_docs)


relevance_query = [0, 1, 0, 0, 1]
number_relevant_docs = 4
k = 3

print("Ejemplo:",recall_at_k(relevance_query, number_relevant_docs, k))

test_1 = [1, 1, 0, 1, 0]
test_2 = [0, 1, 1, 0, 0]

print("Prueba 1:",recall_at_k(test_1, number_relevant_docs=4, k=3))
print("Prueba 2:",recall_at_k(test_2, number_relevant_docs=3, k=4))
print("Edge case - vector vacío:",recall_at_k([], number_relevant_docs=4, k=3))
print("Edge case - k mayor que la longitud:",recall_at_k([1, 0, 1], number_relevant_docs=3, k=5))
print("Edge case - consulta sin documentos relevantes:",recall_at_k([0, 0, 0], number_relevant_docs=0, k=3))

Ejemplo: 0.25
Prueba 1: 0.5
Prueba 2: 0.6666666666666666
Edge case - vector vacío: 0.0
Edge case - k mayor que la longitud: 0.0
Edge case - consulta sin documentos relevantes: 0.0


### Average precision

In [36]:
def average_precision(l: list, f=precision_at_k) -> float:
    """
    Para cada posición k que contiene un documento relevante se calcula
    Precision@K. Average Precision (AP) corresponde al promedio de dichas
    precisiones.

    Parameters
    ----------
    l : list
        Vector binario.

    f : callable
        Función utilizada para calcular Precision@K. Por defecto se utiliza
        precision_at_k.

    Returns
    -------
    float
        AP calculada como el promedio de Precision@K en las posiciones
        donde aparecen documentos relevantes.

        Retorna 0.0 para los siguientes edge cases:
        - el vector está vacío;
        - no hay documentos relevantes.

    """
    l = np.asarray(l, dtype=float)

    if l.size == 0:
        return 0.0

    relevant_positions = np.where(l > 0)[0]

    if relevant_positions.size == 0:
        return 0.0

    precisions = np.array([  f(l, int(position + 1)) for position in relevant_positions],dtype=float)

    return float(np.mean(precisions))


relevance_query = [1, 0, 1, 1, 0, 0, 1, 0]

print("Ejemplo dado:", average_precision(relevance_query))

test_1 = [1, 1, 0, 1, 0]
test_2 = [0, 1, 0, 0, 1]

print("Prueba 1:", average_precision(test_1))
print("Prueba 2:", average_precision(test_2))
print("Edge case - vector vacío:", average_precision([]))
print("Edge case - sin documentos relevantes:",average_precision([0, 0, 0, 0]))

Ejemplo dado: 0.7470238095238095
Prueba 1: 0.9166666666666666
Prueba 2: 0.45
Edge case - vector vacío: 0.0
Edge case - sin documentos relevantes: 0.0


### Mean average precision (MAP)

In [45]:
def mean_average_precision(l: list[list], f=average_precision) -> float:
    """
    Mean Average Precision (MAP) corresponde al promedio del Average Precision obtenido para un conjunto de consultas:

        MAP = (1 / Q) * sum(AP(q))

    donde Q es el número total de consultas.

    Parameters
    ----------
    l : list[list]
        Colección de vectores binarios, uno por consulta.

    f : callable
        Función utilizada para calcular Average Precision. Por defecto se utiliza average_precision.

    Returns
    -------
    float
        MAP calculado como el promedio de los valores de Average Precision obtenidos para las consultas.

        Retorna 0.0 para el siguiente edge case:
        - el conjunto de consultas está vacío.

    """
    if len(l) == 0:
        return 0.0

    result_ap = np.array([f(query) for query in l], dtype=float)

    return float(np.mean(result_ap))


relevance_query_1 = [1, 0, 1, 1, 0, 0, 1, 0]
relevance_query_2 = [0, 1, 1, 0, 1]

relevance_queries = [relevance_query_1, relevance_query_2]

print("Ejemplo:", mean_average_precision(relevance_queries))


test_1 = [[1, 1, 0, 1], [0, 1, 0, 1]]
test_2 = [[1, 0, 0], [1, 1, 1], [0, 0, 1]]

print("Prueba 1:", mean_average_precision(test_1))
print("Prueba 2:", mean_average_precision(test_2))
print("Edge case - conjunto de consultas vacío:", mean_average_precision([]))
print("Caso - consultas sin documentos relevantes:", mean_average_precision([[0, 0, 0], [0, 0, 0]]))
print("Caso - vector vacío dentro del conjunto:", mean_average_precision([[], [1, 0, 1]]))

Ejemplo: 0.6679563492063492
Prueba 1: 0.7083333333333333
Prueba 2: 0.7777777777777778
Edge case - conjunto de consultas vacío: 0.0
Caso - consultas sin documentos relevantes: 0.0
Caso - vector vacío dentro del conjunto: 0.41666666666666663


### DCG at K

In [46]:
def dcg_at_k(l: list, k: int, gain: str) -> float:
    """
    Calcula Discounted Cumulative Gain (DCG) en las primeras k posiciones.

    DCG utiliza los grados de relevancia y aplica un descuento logarítmico
    según la posición del documento. Se admiten dos esquemas de ganancia:
    lineal y exponencial.

    Parameters
    ----------
    l : list
        Vector de relevancia graduada.
    k : int
        Número de primeras posiciones del ranking que se evaluarán.
    gain : str
        Esquema de ganancia. Debe ser "linear" o "exponential".

    Returns
    -------
    float
        DCG@K. Retorna 0.0 si el vector está vacío, si k es menor o igual que
        cero, si k es mayor que la longitud del vector.

    Raises
    ------
    ValueError
        Si gain no corresponde a "linear" o "exponential".
    """
    l = np.asarray(l, dtype=float)

    if l.size == 0 or k <= 0 or k > l.size:
        return 0.0

    positions = np.arange(1, k + 1, dtype=float)
    discounts = np.log2(positions + 1)

    if gain == "linear": gains = l[:k]
    elif gain == "exponential": gains = np.power(2, l[:k]) - 1
    else: raise ValueError("gain debe ser 'linear' o 'exponential'")

    return float(np.sum(gains / discounts))


relevance_query = [3, 2, 3, 0, 1, 2, 3, 0, 0, 1]
k = 5

print("Ejemplo - linear:", dcg_at_k(relevance_query, k, gain="linear"))
print("Ejemplo - exponential:", dcg_at_k(relevance_query, k, gain="exponential"))


test_1 = [3, 2, 1, 0]
test_2 = [0, 1, 2, 3]

print("Prueba 1 - linear:", dcg_at_k(test_1, 4, gain="linear"))
print("Prueba 2 - linear:", dcg_at_k(test_2, 3, gain="linear"))
print("Edge case - vector vacío:", dcg_at_k([], 3, gain="linear"))
print("Edge case - k mayor que la longitud:", dcg_at_k([3, 2, 1], 5, gain="linear"))
print("Edge case - consulta sin documentos relevantes:", dcg_at_k([0, 0, 0], 3, gain="linear"))


Ejemplo - linear: 6.148712314377457
Ejemplo - exponential: 12.779642067948915
Prueba 1 - linear: 4.7618595071429155
Prueba 2 - linear: 1.6309297535714575
Edge case - vector vacío: 0.0
Edge case - k mayor que la longitud: 0.0
Edge case - consulta sin documentos relevantes: 0.0


### NDCG at K

In [49]:

def ndcg_at_k(l: list, k: int, gain: str) -> float:
    """
    Calcula Normalized Discounted Cumulative Gain (NDCG) en k posiciones.

    NDCG compara el DCG observado con el DCG del ordenamiento ideal de los
    mismos grados de relevancia. El resultado queda normalizado con respecto
    al mejor ranking posible.

    Parameters
    ----------
    l : list
        Vector de relevancia graduada.
    k : int
        Número de primeras posiciones del ranking que se evaluarán.
    gain : str
        Esquema de ganancia usado por DCG: "linear" o "exponential".

    Returns
    -------
    float
        NDCG@K. Retorna 0.0 si el vector está vacío, si k es menor o igual que
        cero, si k es mayor que la longitud del vector, si la consulta no
        contiene documentos relevantes o si el DCG ideal es cero.

    Raises
    ------
    ValueError
        Si gain no corresponde a "linear" o "exponential".
    """
    l = np.asarray(l, dtype=float)

    if l.size == 0 or k <= 0 or k > l.size:
        return 0.0

    dcg = dcg_at_k(l, k, gain)
    ideal_l = np.sort(l)[::-1]
    idcg = dcg_at_k(ideal_l, k, gain)

    if idcg == 0:
        return 0.0

    return float(dcg / idcg)


relevance_query = [3, 2, 3, 0, 1, 2, 3, 0, 0, 1]
k = 5

print("Ejemplo - linear:", ndcg_at_k(relevance_query, k, gain="linear"))
print("Ejemplo - exponential:", ndcg_at_k(relevance_query, k, gain="exponential"))

test_1 = [3, 2, 1, 0]
test_2 = [0, 1, 2, 3]

print("Prueba 1 - linear:", ndcg_at_k(test_1, 4, gain="linear"))
print("Prueba 2 - exponential:", ndcg_at_k(test_2, 4, gain="exponential"))
print("Edge case - vector vacío:", ndcg_at_k([], 3, gain="linear"))
print("Edge case - k mayor que la longitud:", ndcg_at_k([3, 2, 1], 5, gain="linear"))
print("Edge case - consulta sin documentos relevantes:", ndcg_at_k([0, 0, 0], 3, gain="linear"))


Ejemplo - linear: 0.76592286264237
Ejemplo - exponential: 0.7357689654680096
Prueba 1 - linear: 1.0
Prueba 2 - exponential: 0.547831481922746
Edge case - vector vacío: 0.0
Edge case - k mayor que la longitud: 0.0
Edge case - consulta sin documentos relevantes: 0.0
